# Train Swin Tiny on PlantVillage (38 classes)

Full fine-tuning with AdamW, cosine LR, MixUp/CutMix, early stopping.

**Instructions:**
1. Runtime > Change runtime type > GPU (T4)
2. Run all cells — dataset downloads automatically via kagglehub
3. Checkpoints saved to Google Drive


In [ ]:
!pip install -q timm scikit-learn tqdm tensorboard kagglehub

## Mount Google Drive (for saving checkpoints)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Download PlantVillage Dataset

In [ ]:
import kagglehub
import os

path = kagglehub.dataset_download('vipoooool/new-plant-diseases-dataset')
print("Dataset path:", path)

# Find the actual data directories
for root, dirs, files in os.walk(path):
    if 'train' in dirs or 'valid' in dirs:
        DATA_ROOT = root
        break
print("Data root:", DATA_ROOT)
print("Contents:", os.listdir(DATA_ROOT))

# Check class count
train_dir = os.path.join(DATA_ROOT, "train")
valid_dir = os.path.join(DATA_ROOT, "valid")
train_classes = sorted(os.listdir(train_dir))
print(f"Train classes: {len(train_classes)}")
print(f"Train images: {sum(len(os.listdir(os.path.join(train_dir, c))) for c in train_classes)}")
valid_classes = sorted(os.listdir(valid_dir))
print(f"Valid images: {sum(len(os.listdir(os.path.join(valid_dir, c))) for c in valid_classes)}")


## Configuration

In [ ]:
import json

CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoints/swin_tiny"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

MODEL_NAME = "swin_tiny_patch4_window7_224"
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 50
LR = 0.0005
WEIGHT_DECAY = 0.0001
SCHEDULER = "cosine"
WARMUP_EPOCHS = 5
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.2
CUTMIX_PROB = 0.5
AUGMENT_LEVEL = "medium"
DROPOUT = 0.3
EARLY_STOPPING_PATIENCE = 10
GRADIENT_CLIP = 1.0
SEED = 789
NUM_CLASSES = 38


## Dataset & DataLoaders

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import transforms, datasets
from collections import Counter

torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

def get_train_transforms(img_size, level):
    base = [transforms.Resize((img_size+32, img_size+32)), transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(0.5)]
    if level == "medium":
        base += [transforms.RandomVerticalFlip(0.3), transforms.RandomRotation(15),
                 transforms.ColorJitter(0.2,0.2,0.2,0.1)]
    elif level == "heavy":
        base += [transforms.RandomVerticalFlip(0.3), transforms.RandomRotation(30),
                 transforms.ColorJitter(0.3,0.3,0.3,0.15), transforms.RandAugment(2,9)]
    base += [transforms.ToTensor(), transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]
    if level in ("medium","heavy"):
        base.append(transforms.RandomErasing(p=0.25 if level=="medium" else 0.4))
    return transforms.Compose(base)

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = datasets.ImageFolder(train_dir, get_train_transforms(IMG_SIZE, AUGMENT_LEVEL))
val_ds = datasets.ImageFolder(valid_dir, val_transform)

# Balanced sampler for class imbalance
counts = Counter(train_ds.targets)
weights = [1.0/counts[t] for t in train_ds.targets]
sampler = WeightedRandomSampler(weights, len(train_ds), replacement=True)

train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4, pin_memory=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Classes: {len(train_ds.classes)}")


## Model

In [ ]:
import timm

class BaseModel(nn.Module):
    def __init__(self, model_name, num_classes=38, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, drop_rate=dropout)
        with torch.no_grad():
            feat_dim = self.backbone(torch.randn(1,3,IMG_SIZE,IMG_SIZE)).shape[-1]
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(feat_dim, num_classes))
        print(f"Model: {model_name}, feat_dim={feat_dim}, params={sum(p.numel() for p in self.parameters())/1e6:.1f}M")

    def forward(self, x):
        return self.head(self.backbone(x))

model = BaseModel(MODEL_NAME, NUM_CLASSES, True, DROPOUT).to(device)


## MixUp/CutMix & Loss

In [ ]:
class MixUpCutMix:
    def __init__(self, mixup_alpha=0.2, cutmix_prob=0.5, num_classes=38):
        self.mixup_alpha = mixup_alpha
        self.cutmix_prob = cutmix_prob
        self.num_classes = num_classes

    def __call__(self, images, targets):
        if random.random() < self.cutmix_prob:
            return self._cutmix(images, targets)
        return self._mixup(images, targets)

    def _mixup(self, images, targets):
        lam = np.random.beta(self.mixup_alpha, self.mixup_alpha) if self.mixup_alpha > 0 else 1.0
        idx = torch.randperm(images.size(0), device=images.device)
        mixed = lam * images + (1-lam) * images[idx]
        t_oh = torch.nn.functional.one_hot(targets, self.num_classes).float()
        return mixed, lam * t_oh + (1-lam) * t_oh[idx]

    def _cutmix(self, images, targets):
        lam = np.random.beta(1.0, 1.0)
        idx = torch.randperm(images.size(0), device=images.device)
        _,_,H,W = images.shape
        cut_r = np.sqrt(1.0-lam); cw,ch = int(W*cut_r),int(H*cut_r)
        cx,cy = random.randint(0,W), random.randint(0,H)
        x1,y1 = max(0,cx-cw//2), max(0,cy-ch//2)
        x2,y2 = min(W,cx+cw//2), min(H,cy+ch//2)
        images[:,:,y1:y2,x1:x2] = images[idx,:,y1:y2,x1:x2]
        lam = 1-(x2-x1)*(y2-y1)/(W*H)
        t_oh = torch.nn.functional.one_hot(targets, self.num_classes).float()
        return images, lam*t_oh + (1-lam)*t_oh[idx]

mixup_fn = MixUpCutMix(MIXUP_ALPHA, CUTMIX_PROB, NUM_CLASSES)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)


## Training

In [ ]:
import time
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

def train_one_epoch(model, loader, criterion, optimizer, scaler, mixup_fn, device, epoch):
    model.train()
    total_loss = 0; all_preds, all_labels = [], []
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        use_mix = MIXUP_ALPHA > 0 and np.random.random() < 0.5
        if use_mix:
            images, labels_mixed = mixup_fn(images, labels); is_mixed = True
        else:
            is_mixed = False
        optimizer.zero_grad()
        with autocast(enabled=torch.cuda.is_available()):
            logits = model(images)
            if is_mixed:
                loss = -(labels_mixed * torch.log_softmax(logits, dim=1)).sum(dim=1).mean()
            else:
                loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        if GRADIENT_CLIP > 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP)
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item()
        if not is_mixed:
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    return total_loss/len(loader), f1_score(all_labels, all_preds, average="macro", zero_division=0) if all_labels else 0

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0; all_preds, all_labels, all_logits = [], [], []
    for images, labels in tqdm(loader, desc="Validating"):
        images, labels = images.to(device), labels.to(device)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item()
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_logits.append(logits.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    acc = np.mean(np.array(all_preds)==np.array(all_labels))
    return total_loss/len(loader), f1, acc, np.concatenate(all_logits), np.array(all_labels)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
if SCHEDULER == "cosine":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS-WARMUP_EPOCHS)
else:
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=LR, epochs=EPOCHS, steps_per_epoch=len(train_dl))
warmup_sched = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, total_iters=WARMUP_EPOCHS) if WARMUP_EPOCHS > 0 else None
scaler = GradScaler(enabled=torch.cuda.is_available())

best_f1 = 0; patience_counter = 0; history = []
print(f"Training {MODEL_NAME} for {EPOCHS} epochs on {device}...")
print("-"*80)

for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    train_loss, train_f1 = train_one_epoch(model, train_dl, criterion, optimizer, scaler, mixup_fn, device, epoch)
    val_loss, val_f1, val_acc, val_logits, val_labels = validate(model, val_dl, criterion, device)
    if warmup_sched and epoch <= WARMUP_EPOCHS:
        warmup_sched.step()
    elif SCHEDULER == "cosine":
        scheduler.step()
    lr = optimizer.param_groups[0]["lr"]
    elapsed = time.time()-t0
    marker = ""
    if val_f1 > best_f1:
        best_f1 = val_f1; patience_counter = 0; marker = " \u2605 BEST"
        torch.save({
            "epoch": epoch, "model_state_dict": model.state_dict(),
            "val_f1": val_f1, "val_acc": val_acc, "model_type": "swin_tiny",
            "num_classes": NUM_CLASSES
        }, os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        np.save(os.path.join(CHECKPOINT_DIR, "best_val_logits.npy"), val_logits)
        np.save(os.path.join(CHECKPOINT_DIR, "best_val_labels.npy"), val_labels)
    else:
        patience_counter += 1
    history.append({
        "epoch": epoch, "train_loss": train_loss, "train_f1": train_f1,
        "val_loss": val_loss, "val_f1": val_f1, "val_acc": val_acc, "lr": lr, "time": elapsed
    })
    print(f"Epoch {epoch:2d}/{EPOCHS} | TrainLoss={train_loss:.4f} F1={train_f1:.4f} | ValLoss={val_loss:.4f} F1={val_f1:.4f} Acc={val_acc:.4f} | lr={lr:.6f} | {elapsed:.0f}s{marker}")
    if patience_counter >= EARLY_STOPPING_PATIENCE:
        print(f"Early stopping at epoch {epoch}"); break

with open(os.path.join(CHECKPOINT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)
print(f"\nDone! Best Val F1: {best_f1:.4f}  Checkpoint: {CHECKPOINT_DIR}")


## Evaluation & Per-Class Report

In [ ]:
from sklearn.metrics import classification_report

ckpt = torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt"), map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(val_dl, desc="Evaluating"):
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

eval_f1 = f1_score(all_labels, all_preds, average="macro")
eval_acc = np.mean(np.array(all_preds)==np.array(all_labels))
print(f"Validation Accuracy: {eval_acc:.4f}")
print(f"Validation Macro F1: {eval_f1:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=train_ds.classes, digits=3))

# Save class names
with open(os.path.join(CHECKPOINT_DIR, "class_names.json"), "w") as f:
    json.dump(train_ds.classes, f, indent=2)
print(f"\n\u2713 All artifacts saved to {CHECKPOINT_DIR}")
